# Análisis de Sentimientos - Reviews de Olist Store

Notebook en construcción: preparación de datos, entrenamiento y evaluación (positivo vs. negativo).

## 1. Carga del dataset

In [ ]:
import pandas as pd

df_reviews = pd.read_csv("./datasets/olist_order_reviews_dataset.csv")
print(f"Dataset cargado correctamente: {len(df_reviews)} filas")

## 2. Limpieza y texto completo

In [ ]:
import numpy as np

# Solo con título y/o mensaje
df_text = df_reviews.dropna(
    subset=["review_comment_title", "review_comment_message"], how="all"
).copy()

# Sin neutras (3 estrellas) y target binario: 1 = 4-5 estrellas
df_filtered = df_text[df_text["review_score"] != 3].copy()
df_filtered["target"] = (df_filtered["review_score"] >= 4).astype(int)

# Orden temporal para split sin fuga
df_sorted = df_filtered.sort_values(by="review_creation_date").reset_index(drop=True)

# Une título + mensaje: quita puntuación final, une con ". " y normaliza espacios
title = (
    df_sorted["review_comment_title"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.rstrip(".!?:;")
)

message = df_sorted["review_comment_message"].fillna("").astype(str).str.strip()
separador = np.where((title != "") & (message != ""), ". ", "")
df_sorted["review_text_full"] = (
    (title + separador + message).str.replace(r"\s+", " ", regex=True).str.strip().str.lower()
)

print(f"Total de comentarios con títulos y/o mensajes: {len(df_sorted["review_text_full"])}")

# Features / target
X = df_sorted["review_text_full"]
y = df_sorted["target"]